In [23]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Dict
import pandas as pd


class Alert(ABC):

    @property
    @abstractmethod
    def id(self) -> int: ...

    @abstractmethod
    def should_raise(self) -> bool: ...

    @abstractmethod
    def describe(self) -> str: ...


@dataclass
class Pathogen:
    org_id: Optional[int]
    org_name: str
    danger_weight: float
    time_window_days: int
    ward_thresholds: Dict[str, int] 
    staff_threshold: int

    def get_ward_threshold(self, ward_size: int) -> int:
        if ward_size <= 5:
            return self.ward_thresholds["5"]
        elif ward_size <= 10:
            return self.ward_thresholds["10"]
        else:
            return self.ward_thresholds["20"]


@dataclass
class MicrobiologyAlert(Alert):
    counter_id: int
    pathogen: Pathogen
    ward_id: int
    ward_size: int
    start_time: datetime
    alert_type: str
    curr_patient_number: int

    @property
    def id(self) -> int:
        return self.counter_id

    @property
    def org_id(self) -> Optional[int]:
        return self.pathogen.org_id

    def should_raise(self) -> bool:
        return self.curr_patient_number >= self.pathogen.get_ward_threshold(self.ward_size)

    def describe(self) -> str:
        thresh = self.pathogen.get_ward_threshold(self.ward_size)
        status = "⚠ ALERT" if self.should_raise() else "OK"
        return (
            f"[{status}][{self.alert_type}] Ward {self.ward_id} "
            f"({self.ward_size} beds) | {self.pathogen.org_name} | "
            f"{self.curr_patient_number}/{thresh} patients @ {self.start_time}"
        )


@dataclass
class WardAlert(Alert):
    counter_id: int
    ward_id: int
    pathogen: Pathogen
    ward_size: int
    start_time: datetime
    alert_type: str
    curr_patient_number: int

    @property
    def id(self) -> int:
        return self.counter_id

    def should_raise(self) -> bool:
        return self.curr_patient_number >= self.pathogen.get_ward_threshold(self.ward_size)

    def describe(self) -> str:
        thresh = self.pathogen.get_ward_threshold(self.ward_size)
        status = "⚠ ALERT" if self.should_raise() else "OK"
        return (
            f"[{status}][{self.alert_type}] Ward {self.ward_id} "
            f"({self.ward_size} beds) | {self.pathogen.org_name} | "
            f"{self.curr_patient_number}/{thresh} patients @ {self.start_time}"
        )
from __future__ import annotations
from dataclasses import dataclass
from typing import Optional, Dict
import pandas as pd


@dataclass(frozen=True)
class Pathogen:
    key: str                     # canonical name / registry key
    org_id: Optional[int]
    danger_weight: float
    time_window_days: int
    ward_thresholds: Dict[int, int]  # ward size breakpoint -> threshold
    staff_threshold: int

    def ward_threshold(self, ward_size: int) -> int:
        """Largest breakpoint <= ward_size, fallback to max breakpoint."""
        cutoffs = sorted(self.ward_thresholds.keys())
        chosen = max((c for c in cutoffs if c <= ward_size), default=cutoffs[-1])
        return self.ward_thresholds[chosen]


class PathogenRegistry:
    def __init__(self):
        self._by_key: Dict[str, Pathogen] = {}
        self._by_org_id: Dict[int, Pathogen] = {}

    def register(self, key: str, **kwargs) -> Pathogen:
        key_norm = key.upper()
        pathogen = Pathogen(key=key_norm, **kwargs)
        self._by_key[key_norm] = pathogen
        if pathogen.org_id is not None:
            self._by_org_id[pathogen.org_id] = pathogen
        return pathogen

    def get(self, key: str) -> Optional[Pathogen]:
        return self._by_key.get(key.upper())

    def get_by_org_id(self, org_id: int) -> Optional[Pathogen]:
        return self._by_org_id.get(org_id)

    def __contains__(self, key: str) -> bool:
        return key.upper() in self._by_key

    def __iter__(self):
        return iter(self._by_key.values())

    def as_dataframe(self) -> pd.DataFrame:
        rows = [
            {
                "key": p.key,
                "org_id": p.org_id,
                "danger_weight": p.danger_weight,
                "time_window_days": p.time_window_days,
                "ward_thresholds": p.ward_thresholds,
                "staff_threshold": p.staff_threshold,
            }
            for p in self._by_key.values()
        ]
        return pd.DataFrame(rows).set_index("key").sort_index()


REGISTRY = PathogenRegistry()

# HIGH-RISK (1 case triggers alert - MDR, C. diff, etc.)
REGISTRY.register("CLOSTRIDIUM DIFFICILE", org_id=None, danger_weight=3.0, time_window_days=3, ward_thresholds={5: 1, 10: 1, 20: 1}, staff_threshold=2)
REGISTRY.register("ACINETOBACTER BAUMANNII COMPLEX", org_id=None, danger_weight=3.0, time_window_days=3, ward_thresholds={5: 1, 10: 1, 20: 1}, staff_threshold=2)
REGISTRY.register("ACINETOBACTER BAUMANNII", org_id=None, danger_weight=3.0, time_window_days=3, ward_thresholds={5: 1, 10: 1, 20: 1}, staff_threshold=2)
REGISTRY.register("POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS", org_id=None, danger_weight=3.0, time_window_days=1, ward_thresholds={5: 1, 10: 1, 20: 2}, staff_threshold=1)
REGISTRY.register("PSEUDOMONAS AERUGINOSA", org_id=None, danger_weight=3.0, time_window_days=1, ward_thresholds={5: 1, 10: 1, 20: 2}, staff_threshold=1)
REGISTRY.register("KLEBSIELLA PNEUMONIAE", org_id=None, danger_weight=3.0, time_window_days=1, ward_thresholds={5: 1, 10: 1, 20: 1}, staff_threshold=1)
REGISTRY.register("ESCHERICHIA COLI", org_id=None, danger_weight=3.0, time_window_days=3, ward_thresholds={5: 1, 10: 1, 20: 2}, staff_threshold=1)
REGISTRY.register("STAPH AUREUS COAG +", org_id=None, danger_weight=3.0, time_window_days=1, ward_thresholds={5: 1, 10: 1, 20: 2}, staff_threshold=1)
REGISTRY.register("PROVIDENCIA STUARTII", org_id=None, danger_weight=2.5, time_window_days=2, ward_thresholds={5: 1, 10: 1, 20: 2}, staff_threshold=2)
REGISTRY.register("STENOTROPHOMONAS (XANTHOMONAS) MALTOPHILIA", org_id=None, danger_weight=2.5, time_window_days=2, ward_thresholds={5: 1, 10: 1, 20: 2}, staff_threshold=2)

# MEDIUM-RISK (2 cases - Enterobacteriaceae, fungi, anaerobes)
REGISTRY.register("GRAM NEGATIVE ROD(S)", org_id=None, danger_weight=1.5, time_window_days=2, ward_thresholds={5: 1, 10: 1, 20: 2}, staff_threshold=3)
REGISTRY.register("GRAM NEGATIVE ROD #1", org_id=None, danger_weight=1.5, time_window_days=2, ward_thresholds={5: 1, 10: 1, 20: 2}, staff_threshold=3)
REGISTRY.register("GRAM NEGATIVE ROD #2", org_id=None, danger_weight=2.0, time_window_days=2, ward_thresholds={5: 1, 10: 2, 20: 3}, staff_threshold=2)
REGISTRY.register("GRAM NEGATIVE ROD #3", org_id=None, danger_weight=2.0, time_window_days=2, ward_thresholds={5: 1, 10: 2, 20: 3}, staff_threshold=2)
REGISTRY.register("GRAM NEGATIVE ROD #4", org_id=None, danger_weight=2.0, time_window_days=2, ward_thresholds={5: 1, 10: 2, 20: 3}, staff_threshold=2)
REGISTRY.register("ENTEROBACTERIACEAE", org_id=None, danger_weight=2.0, time_window_days=2, ward_thresholds={5: 2, 10: 2, 20: 3}, staff_threshold=3)
REGISTRY.register("KLEBSIELLA OXYTOCA", org_id=None, danger_weight=2.5, time_window_days=2, ward_thresholds={5: 1, 10: 2, 20: 3}, staff_threshold=2)
REGISTRY.register("PROTEUS MIRABILIS", org_id=None, danger_weight=1.5, time_window_days=1, ward_thresholds={5: 2, 10: 3, 20: 4}, staff_threshold=3)
REGISTRY.register("PROTEUS SPECIES", org_id=None, danger_weight=1.5, time_window_days=1, ward_thresholds={5: 2, 10: 3, 20: 4}, staff_threshold=3)
REGISTRY.register("SERRATIA MARCESCENS", org_id=None, danger_weight=2.5, time_window_days=2, ward_thresholds={5: 1, 10: 2, 20: 3}, staff_threshold=2)
REGISTRY.register("MORGANELLA MORGANII", org_id=None, danger_weight=2.0, time_window_days=2, ward_thresholds={5: 2, 10: 2, 20: 3}, staff_threshold=3)
REGISTRY.register("PROVIDENCIA RETTGERI", org_id=None, danger_weight=2.0, time_window_days=2, ward_thresholds={5: 2, 10: 2, 20: 3}, staff_threshold=3)
REGISTRY.register("ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER", org_id=None, danger_weight=1.5, time_window_days=2, ward_thresholds={5: 1, 10: 1, 20: 2}, staff_threshold=2)
REGISTRY.register("MYCELIA STERILIA", org_id=None, danger_weight=1.0, time_window_days=4, ward_thresholds={5: 2, 10: 3, 20: 4}, staff_threshold=4)
REGISTRY.register("BACTEROIDES FRAGILIS GROUP", org_id=None, danger_weight=1.5, time_window_days=3, ward_thresholds={5: 2, 10: 2, 20: 3}, staff_threshold=3)
REGISTRY.register("ANAEROBIC GRAM POSITIVE ROD(S)", org_id=None, danger_weight=1.5, time_window_days=3, ward_thresholds={5: 2, 10: 2, 20: 3}, staff_threshold=3)

# LOW-RISK (3+ cases - colonizers, skin flora)
REGISTRY.register("STAPHYLOCOCCUS, COAGULASE NEGATIVE", org_id=None, danger_weight=1.5, time_window_days=1, ward_thresholds={5: 2, 10: 2, 20: 3}, staff_threshold=2)
REGISTRY.register("YEAST", org_id=None, danger_weight=1.0, time_window_days=3, ward_thresholds={5: 1, 10: 1, 20: 3}, staff_threshold=4)
REGISTRY.register("CANDIDA ALBICANS, PRESUMPTIVE IDENTIFICATION", org_id=None, danger_weight=1.0, time_window_days=2, ward_thresholds={5: 1, 10: 1, 20: 2}, staff_threshold=3)
REGISTRY.register("STREPTOCOCCUS SPECIES", org_id=None, danger_weight=1.0, time_window_days=2, ward_thresholds={5: 1, 10: 2, 20: 3}, staff_threshold=4)
REGISTRY.register("STREPTOCOCCUS PNEUMONIAE", org_id=None, danger_weight=1.5, time_window_days=2, ward_thresholds={5: 1, 10: 2, 20: 3}, staff_threshold=3)
REGISTRY.register("ENTEROCOCCUS FAECALIS", org_id=None, danger_weight=1.5, time_window_days=2, ward_thresholds={5: 2, 10: 2, 20: 3}, staff_threshold=3)
REGISTRY.register("ENTEROCOCCUS FAECIUM", org_id=None, danger_weight=2.0, time_window_days=2, ward_thresholds={5: 2, 10: 2, 20: 3}, staff_threshold=3)
REGISTRY.register("ENTEROCOCCUS SP.", org_id=None, danger_weight=1.5, time_window_days=2, ward_thresholds={5: 2, 10: 2, 20: 3}, staff_threshold=3)
REGISTRY.register("PROBABLE ENTEROCOCCUS", org_id=None, danger_weight=1.5, time_window_days=2, ward_thresholds={5: 2, 10: 2, 20: 3}, staff_threshold=3)
REGISTRY.register("ALPHA STREPTOCOCCI", org_id=None, danger_weight=1.0, time_window_days=2, ward_thresholds={5: 2, 10: 3, 20: 4}, staff_threshold=4)
REGISTRY.register("BETA STREPTOCOCCUS GROUP B", org_id=None, danger_weight=1.5, time_window_days=2, ward_thresholds={5: 2, 10: 2, 20: 3}, staff_threshold=3)
REGISTRY.register("PRESUMPTIVE STREPTOCOCCUS BOVIS", org_id=None, danger_weight=1.5, time_window_days=2, ward_thresholds={5: 2, 10: 2, 20: 3}, staff_threshold=3)
REGISTRY.register("GRAM POSITIVE COCCUS(COCCI)", org_id=None, danger_weight=1.0, time_window_days=2, ward_thresholds={5: 2, 10: 3, 20: 4}, staff_threshold=4)
REGISTRY.register("GRAM POSITIVE RODS", org_id=None, danger_weight=1.5, time_window_days=2, ward_thresholds={5: 2, 10: 3, 20: 4}, staff_threshold=3)
REGISTRY.register("CORYNEBACTERIUM SPECIES (DIPHTHEROIDS)", org_id=None, danger_weight=2.0, time_window_days=2, ward_thresholds={5: 1, 10: 2, 20: 3}, staff_threshold=2)
REGISTRY.register("CORYNEBACTERIUM STRIATUM", org_id=None, danger_weight=2.0, time_window_days=2, ward_thresholds={5: 1, 10: 2, 20: 3}, staff_threshold=2)
REGISTRY.register("LACTOBACILLUS SPECIES", org_id=None, danger_weight=1.0, time_window_days=3, ward_thresholds={5: 3, 10: 4, 20: 5}, staff_threshold=4)
REGISTRY.register("MYCOBACTERIUM AVIUM COMPLEX", org_id=None, danger_weight=1.5, time_window_days=1, ward_thresholds={5: 1, 10: 2, 20: 4}, staff_threshold=3)
REGISTRY.register("GRAM POSITIVE BACTERIA", org_id=None, danger_weight=1.0, time_window_days=2, ward_thresholds={5: 2, 10: 3, 20: 4}, staff_threshold=4)

# GENERIC / LOW-SIGNAL
REGISTRY.register("ORGANISM", org_id=None, danger_weight=0.5, time_window_days=7, ward_thresholds={5: 5, 10: 7, 20: 10}, staff_threshold=5)



Pathogen(key='ORGANISM', org_id=None, danger_weight=0.5, time_window_days=7, ward_thresholds={5: 5, 10: 7, 20: 10}, staff_threshold=5)

In [24]:
REGISTRY.as_dataframe()

,org_id,danger_weight,time_window_days,ward_thresholds,staff_threshold
key,,,,,
ACINETOBACTER BAUMANNII,None,3.0,3,"{5: 1, 10: 1, 20: 1}",2
ACINETOBACTER BAUMANNII COMPLEX,None,3.0,3,"{5: 1, 10: 1, 20: 1}",2
ALPHA STREPTOCOCCI,None,1.0,2,"{5: 2, 10: 3, 20: 4}",4
ANAEROBIC GRAM POSITIVE ROD(S),None,1.5,3,"{5: 2, 10: 2, 20: 3}",3
"ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER",None,1.5,2,"{5: 1, 10: 1, 20: 2}",2
BACTEROIDES FRAGILIS GROUP,None,1.5,3,"{5: 2, 10: 2, 20: 3}",3
BETA STREPTOCOCCUS GROUP B,None,1.5,2,"{5: 2, 10: 2, 20: 3}",3
"CANDIDA ALBICANS, PRESUMPTIVE IDENTIFICATION",None,1.0,2,"{5: 1, 10: 1, 20: 2}",3
CLOSTRIDIUM DIFFICILE,None,3.0,3,"{5: 1, 10: 1, 20: 1}",2


In [25]:
import sqlite3
import pandas as pd

# Assuming db_path and desired_tables are defined earlier, e.g.:
db_path = '/Volumes/T7/OOP_project/OOP_database.db'  # From your MIMIC/OOP project [cite:16]
desired_tables = ['PATIENTS', 'CAREGIVERS', 'D_ITEMS', 'ADMISSIONS', 'ICUSTAYS', 'NOTEEVENTS', 'MICROBIOLOGYEVENTS', 'TRANSFERS', 'OUTPUTEVENTS', 'CHARTEVENTS']

from contextlib import contextmanager

@contextmanager
def get_db_connection(db_path):
    conn = sqlite3.connect(db_path)
    try:
        yield conn
    finally:
        conn.close()

with get_db_connection(db_path) as conn:
    db_tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table'", conn
    )["name"].str.upper().tolist()

for tbl in desired_tables:
    if tbl in db_tables:
        print(f"Table {tbl} exists.")  # Or your logic here
    else:
        print(f"Table {tbl} missing.")


Table PATIENTS exists.
Table CAREGIVERS exists.
Table D_ITEMS exists.
Table ADMISSIONS exists.
Table ICUSTAYS exists.
Table NOTEEVENTS exists.
Table MICROBIOLOGYEVENTS exists.
Table TRANSFERS exists.
Table OUTPUTEVENTS exists.
Table CHARTEVENTS exists.


In [26]:
# Load tables into data dict (run AFTER table check)
data = {}
with get_db_connection(db_path) as conn:  # Reuse your context manager
    for tbl in desired_tables:
        if tbl in db_tables:
            df_name = tbl.upper()  
            data[df_name] = pd.read_sql_query(f"SELECT * FROM {tbl}", conn)
            print(f"Loaded {df_name} ({len(data[df_name])} rows, {data[df_name].shape[1]} cols)")

# Access like your CSV vars:
patients = data['PATIENTS']
caregivers = data[ 'CAREGIVERS']
d_items = data['D_ITEMS']
admissions = data['ADMISSIONS']
icustays = data['ICUSTAYS']
noteevents = data['NOTEEVENTS']
microbiologyevents = data['MICROBIOLOGYEVENTS']
transfers = data['TRANSFERS']
outputevents = data['OUTPUTEVENTS']
chartevents = data['CHARTEVENTS']


Loaded PATIENTS (100 rows, 7 cols)
Loaded CAREGIVERS (7567 rows, 3 cols)
Loaded D_ITEMS (12487 rows, 9 cols)
Loaded ADMISSIONS (129 rows, 18 cols)
Loaded ICUSTAYS (136 rows, 11 cols)
Loaded NOTEEVENTS (0 rows, 11 cols)
Loaded MICROBIOLOGYEVENTS (2003 rows, 16 cols)
Loaded TRANSFERS (524 rows, 13 cols)
Loaded OUTPUTEVENTS (11319 rows, 13 cols)
Loaded CHARTEVENTS (758274 rows, 15 cols)


In [39]:
# pick both columns correctly, stack/flatten and drop NaNs
ward_cols = icustays[['FIRST_WARDID', 'LAST_WARDID']]
wards = pd.unique(ward_cols.stack().dropna())          #   or .values.ravel()

# inspect
print(len(wards), "distinct ward ids")
wards

9 distinct ward ids


array([52, 12, 50, 57,  7, 23, 15, 33, 14])

In [28]:
from datetime import timedelta
from collections import defaultdict

microbiologyevents.columns = microbiologyevents.columns.str.upper()
icustays.columns           = icustays.columns.str.upper()

In [42]:
import pandas as pd
import numpy as np

# ALL WARDS from your data
ALL_WARD_IDS = [52, 12, 50, 57, 7, 23, 15, 33, 14]

# WARD SIZES (5/10/20 buckets - adjust based on your data)
WARD_SIZES = {
    52: 10,
    12: 20,  # assume largest
    50: 10,
    57: 5,   # assume smallest
    7:  20,
    23: 15,  # mid-size -> use 10 or 20 bucket
    15: 10,
    33: 20,
    14: 10,
}

# Pre-compute HADM_IDs per ward
ward_hadm = {}
for ward_id in ALL_WARD_IDS:
    ward_hadm[ward_id] = icustays.loc[
        (icustays["FIRST_WARDID"] == ward_id) | (icustays["LAST_WARDID"] == ward_id),
        "HADM_ID"
    ].dropna().unique()

microbiologyevents["CHARTDATE"] = pd.to_datetime(microbiologyevents["CHARTDATE"], errors="coerce")

# Process ALL wards
all_ward_pos = []
for ward_id in ALL_WARD_IDS:
    print(f"Processing ward {ward_id} (size: {WARD_SIZES[ward_id]})")
    
    ward_pos = microbiologyevents[
        microbiologyevents["HADM_ID"].isin(ward_hadm[ward_id])
        & microbiologyevents["ORG_NAME"].notna()
        & microbiologyevents["CHARTDATE"].notna()
    ].copy()
    
    if len(ward_pos) == 0:
        print(f"  No microbiology data for ward {ward_id}")
        continue
    
    ward_pos["ORG_NAME"] = ward_pos["ORG_NAME"].astype(str).str.upper().str.strip()
    ward_pos["WARD_ID"] = ward_id
    ward_pos["WARD_SIZE"] = WARD_SIZES[ward_id]
    ward_pos = ward_pos.sort_values(["ORG_NAME", "CHARTDATE"])
    
    all_ward_pos.append(ward_pos)
    print(f"  Found {len(ward_pos)} positive cultures")

# Combine all wards
ward_pos_all = pd.concat(all_ward_pos, ignore_index=True) if all_ward_pos else pd.DataFrame()

print(f"\nTotal across {len(ALL_WARD_IDS)} wards: {len(ward_pos_all)} cultures")
print(ward_pos_all.groupby(["WARD_ID", "WARD_SIZE"])["ORG_NAME"].agg(["count"]).round(1))


Processing ward 52 (size: 10)
  Found 277 positive cultures
Processing ward 12 (size: 20)
  Found 10 positive cultures
Processing ward 50 (size: 10)
  Found 147 positive cultures
Processing ward 57 (size: 5)
  Found 220 positive cultures
Processing ward 7 (size: 20)
  Found 166 positive cultures
Processing ward 23 (size: 15)
  Found 160 positive cultures
Processing ward 15 (size: 10)
  Found 42 positive cultures
Processing ward 33 (size: 20)
  Found 174 positive cultures
Processing ward 14 (size: 10)
  Found 64 positive cultures

Total across 9 wards: 1260 cultures
                   count
WARD_ID WARD_SIZE       
7       20           166
12      20            10
14      10            64
15      10            42
23      15           160
33      20           174
50      10           147
52      10           277
57      5            220


In [43]:
# Discharge times for ALL wards
ward_discharges = {}
for ward_id in ALL_WARD_IDS:
    ward_discharges[ward_id] = icustays[
        (icustays["FIRST_WARDID"] == ward_id) | (icustays["LAST_WARDID"] == ward_id)
    ][["SUBJECT_ID", "OUTTIME"]].copy()
    ward_discharges[ward_id]["OUTTIME"] = pd.to_datetime(ward_discharges[ward_id]["OUTTIME"])

# Join discharge times onto positive cultures (per-ward merge)
ward_pos_list = []
for ward_id in ALL_WARD_IDS:
    ward_pos = ward_pos_all[ward_pos_all["WARD_ID"] == ward_id].copy()
    if len(ward_pos) > 0:
        ward_pos = ward_pos.merge(
            ward_discharges[ward_id], 
            on="SUBJECT_ID", 
            how="left",
            suffixes=("", f"_ward{ward_id}")
        )
        ward_pos_list.append(ward_pos)

ward_pos_all = pd.concat(ward_pos_list, ignore_index=True) if ward_pos_list else pd.DataFrame()

# Ensure datetime consistency
ward_pos_all["OUTTIME"] = pd.to_datetime(ward_pos_all["OUTTIME"], errors="coerce")
ward_pos_all["CHARTDATE"] = pd.to_datetime(ward_pos_all["CHARTDATE"], errors="coerce")

print("Discharge merge complete:")
print(ward_pos_all[["WARD_ID", "SUBJECT_ID", "CHARTDATE", "OUTTIME"]].head())


Discharge merge complete:
   WARD_ID  SUBJECT_ID  CHARTDATE             OUTTIME
0       52       41914 2145-12-06 2145-12-14 18:00:12
1       52       41914 2145-12-06 2145-12-14 18:00:12
2       52       41914 2145-12-06 2145-12-14 18:00:12
3       52       41914 2145-12-06 2145-12-14 18:00:12
4       52       41914 2145-12-06 2145-12-14 18:00:12


In [33]:
# convert dates once
microbiologyevents["CHARTDATE"] = pd.to_datetime(
    microbiologyevents["CHARTDATE"], errors="coerce"
)

# make a long table of hadm→ward (first and last)
ward_map = (
    icustays[["HADM_ID", "FIRST_WARDID", "LAST_WARDID"]]
    .melt(id_vars="HADM_ID",
          value_vars=["FIRST_WARDID", "LAST_WARDID"],
          value_name="WARD_ID")
    .dropna(subset=["WARD_ID"])        # toss stays with no ward info
    .drop_duplicates()                # one row per hadm/ward pair
)

# filter cultures to those hadm_ids we care about and clean names
ward_pos = microbiologyevents[
    microbiologyevents["HADM_ID"].isin(ward_map["HADM_ID"])
    & microbiologyevents["ORG_NAME"].notna()
    & microbiologyevents["CHARTDATE"].notna()
].copy()

ward_pos = ward_pos.merge(ward_map, on="HADM_ID", how="left")

ward_pos["ORG_NAME"] = (
    ward_pos["ORG_NAME"].astype(str).str.upper().str.strip()
)

ward_pos = ward_pos.sort_values(
    ["WARD_ID", "ORG_NAME", "CHARTDATE"]
)

In [44]:
ward_pos

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,ITEMID,SPEC_TYPE_DESC,ORG_ITEMID,ORG_NAME,ISOLATE_NUM,AB_ITEMID,AB_NAME,DILUTION_TEXT,DILUTION_COMPARISON,DILUTION_VALUE,INTERPRETATION,WARD_ID,WARD_SIZE,OUTTIME
0,136126,10130,156668,2161-01-30,NaN,70076,TISSUE,80052.0,ALPHA STREPTOCOCCI,1.0,NaN,NaN,NaN,NaN,NaN,NaN,14,10,2161-02-06 12:34:00
1,136125,10130,156668,2161-01-30,NaN,70067,SWAB,80112.0,BACTEROIDES FRAGILIS GROUP,1.0,NaN,NaN,NaN,NaN,NaN,NaN,14,10,2161-02-06 12:34:00
2,136120,10130,156668,2161-01-30,NaN,70076,TISSUE,80045.0,BETA STREPTOCOCCUS GROUP B,1.0,NaN,NaN,NaN,NaN,NaN,NaN,14,10,2161-02-06 12:34:00
3,136121,10130,156668,2161-01-30,NaN,70067,SWAB,80045.0,BETA STREPTOCOCCUS GROUP B,1.0,NaN,NaN,NaN,NaN,NaN,NaN,14,10,2161-02-06 12:34:00
4,136105,10127,182839,2198-07-14,2198-07-14 04:34:00,70079,URINE,80254.0,"CANDIDA ALBICANS, PRESUMPTIVE IDENTIFICATION",1.0,NaN,NaN,NaN,NaN,NaN,NaN,14,10,2198-07-20 14:56:23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,136090,10127,182839,2198-07-09,2198-07-09 19:40:00,70062,SPUTUM,80087.0,STENOTROPHOMONAS (XANTHOMONAS) MALTOPHILIA,1.0,90008.0,TRIMETHOPRIM/SULFA,<=1,<=,1.0,S,14,10,2198-07-20 14:56:23
60,136104,10127,182839,2198-07-14,2198-07-14 04:34:00,70062,SPUTUM,80087.0,STENOTROPHOMONAS (XANTHOMONAS) MALTOPHILIA,1.0,NaN,NaN,NaN,NaN,NaN,NaN,14,10,2198-07-20 14:56:23
61,136108,10127,182839,2198-07-14,2198-07-14 11:35:00,70021,BRONCHOALVEOLAR LAVAGE,80087.0,STENOTROPHOMONAS (XANTHOMONAS) MALTOPHILIA,2.0,90008.0,TRIMETHOPRIM/SULFA,<=1,<=,1.0,S,14,10,2198-07-20 14:56:23
62,136109,10127,182839,2198-07-14,2198-07-14 11:35:00,70021,BRONCHOALVEOLAR LAVAGE,80087.0,STENOTROPHOMONAS (XANTHOMONAS) MALTOPHILIA,1.0,90008.0,TRIMETHOPRIM/SULFA,<=1,<=,1.0,S,14,10,2198-07-20 14:56:23


In [45]:
ward_hadm

{52: array([198503, 157609, 177678, 142345, 155297, 186361, 172454, 195911,
        164869, 188574, 198480, 109698, 114867, 167754, 172082, 171878,
        101361, 174863, 130681, 140372, 124073, 129273, 151798, 175237,
        100375, 176016, 197611, 118192, 142539, 176805, 167612]),
 12: array([114648, 133283, 170119, 189483, 183314, 168674, 157839, 149044,
        105150, 199395]),
 50: array([126949, 174245, 127703, 159647, 126002, 170883, 180391, 174997,
        152032, 198480, 163189, 121860, 161765, 134993, 126179, 180546,
        128293]),
 57: array([126949, 165436, 100969, 133110, 103379, 145203, 167181, 186071,
        167957, 179418, 130870, 190301, 160445, 139932, 193924, 116518,
        125449, 146672]),
 7: array([199207, 186361, 125013, 142633, 192189, 121860, 149950, 168233,
        154156, 134993, 146893, 182879, 131048, 112662, 198330, 168803,
        170883, 151323]),
 23: array([173269, 102203, 142582, 138132, 149469, 160442, 170883, 168074,
        122098, 182664,

In [46]:
episodes_all = []

for org_name, pathogen in REGISTRY._by_key.items():  # iterate registry
    ward_data = ward_pos_all[ward_pos_all["ORG_NAME"] == org_name].copy()
    if ward_data.empty:
        continue

    # Group by WARD_ID first (multi-ward support)
    ward_episodes = []
    for ward_id, ward_group in ward_data.groupby("WARD_ID"):
        df = ward_group.sort_values("CHARTDATE").reset_index(drop=True)
        ward_size = ward_group["WARD_SIZE"].iloc[0]

        # Episode logic (same as yours, but per-ward)
        episode_ids = []
        episode_id = 0
        episode_start = None
        episode_end = None

        for _, row in df.iterrows():
            if episode_start is None:
                episode_start = row["CHARTDATE"]
                episode_end = row["OUTTIME"]
                episode_ids.append(episode_id)
            else:
                still_occupied = pd.notna(episode_end) and pd.notna(row["CHARTDATE"]) and row["CHARTDATE"] <= episode_end
                gap = (row["CHARTDATE"] - episode_start).total_seconds() / 86400 if pd.notna(row["CHARTDATE"]) and pd.notna(episode_start) else float('inf')

                if still_occupied or gap <= pathogen.time_window_days:
                    # Extend episode
                    if pd.notna(row["OUTTIME"]):
                        episode_end = max(episode_end, row["OUTTIME"]) if pd.notna(episode_end) else row["OUTTIME"]
                else:
                    # New episode
                    episode_id += 1
                    episode_start = row["CHARTDATE"]
                    episode_end = row["OUTTIME"]

                episode_ids.append(episode_id)

        df["EPISODE_ID"] = episode_ids

        # Aggregate episodes
        ep = df.groupby("EPISODE_ID").agg(
            start_time=("CHARTDATE", "min"),
            end_time=("OUTTIME", "max"),
            culture_events=("ROW_ID", "count"),
            unique_patients=("SUBJECT_ID", "nunique"),
        ).reset_index(drop=True)

        ep["ward_id"] = ward_id
        ep["ward_size"] = ward_size
        ep["org_name"] = pathogen.key
        ep["danger_weight"] = pathogen.danger_weight
        ep["window_days"] = pathogen.time_window_days
        ep["threshold"] = pathogen.ward_threshold(ward_size)  # ← dynamic per ward size!
        ep["alert"] = ep["unique_patients"] >= ep["threshold"]

        ward_episodes.append(ep)

    # Combine ward episodes for this pathogen
    if ward_episodes:
        pathogen_ep = pd.concat(ward_episodes, ignore_index=True)
        episodes_all.append(pathogen_ep)

episodes_df = pd.concat(episodes_all, ignore_index=True) if episodes_all else pd.DataFrame()

print(f"Generated {len(episodes_df)} episodes across {episodes_df['ward_id'].nunique()} wards")
print(episodes_df.groupby("ward_id")["alert"].sum().sort_values(ascending=False))


Generated 195 episodes across 9 wards
ward_id
52    24
57    21
23    18
50    13
14     8
15     4
7      2
12     1
33     1
Name: alert, dtype: int64


In [47]:
if episodes_df.empty:
    print("No pathogen episodes found in ward 52 for the registry organisms.")
else:
    print("Episodes (counters) found:", len(episodes_df))
    print("Alerts fired (episodes where unique_patients >= threshold):", int(episodes_df["alert"].sum()))

    print("\nAlerts per pathogen:")
    print(
        episodes_df[episodes_df["alert"]]
        .groupby("org_name")
        .agg(
            alert_episodes=("alert", "size"),
            first_alert=("start_time", "min"),
            last_alert=("end_time", "max"),
            max_patients=("unique_patients", "max"),
            threshold=("threshold", "first"),
            window_days=("window_days", "first"),
        )
        .sort_values("alert_episodes", ascending=False)
        .to_string()
    )

    print("\nAll ALERT episodes (each row = one counter instance):")
    print(
        episodes_df[episodes_df["alert"]]
        .sort_values(["org_name", "start_time"])
        .to_string(index=False)
    )


Episodes (counters) found: 195
Alerts fired (episodes where unique_patients >= threshold): 92

Alerts per pathogen:
                                                 alert_episodes first_alert          last_alert  max_patients  threshold  window_days
org_name                                                                                                                             
YEAST                                                        23  2107-01-05 2202-10-04 17:07:38             1          1            3
STAPH AUREUS COAG +                                          13  2111-01-12 2202-10-04 17:07:38             1          1            1
POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS              10  2127-07-29 2202-10-04 17:07:38             1          1            1
ESCHERICHIA COLI                                              8  2107-01-29 2170-12-19 19:33:09             1          1            3
PSEUDOMONAS AERUGINOSA                                        7  2112-02-05 2202

In [52]:
with get_db_connection(db_path) as conn:
    cursor = conn.cursor()
    
    # Enhanced EPISODES table (stores ALL episodes, flagged as alert or not)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS MICRO_EPISODES (
            EPISODE_ID INTEGER PRIMARY KEY AUTOINCREMENT,
            WARD_ID INTEGER NOT NULL,
            ORG_NAME TEXT NOT NULL,
            ORG_ID INTEGER,
            START_TIME TIMESTAMP NOT NULL,
            END_TIME TIMESTAMP,
            NUM_PATIENTS INTEGER NOT NULL,
            CULTURE_EVENTS INTEGER NOT NULL,
            THRESHOLD INTEGER NOT NULL,
            IS_ALERT BOOLEAN NOT NULL,
            DANGER_WEIGHT REAL NOT NULL,
            WINDOW_DAYS INTEGER NOT NULL,
            WARD_SIZE INTEGER NOT NULL,
            CREATED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    # Indexes
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_episodes_ward_time ON MICRO_EPISODES(WARD_ID, START_TIME)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_episodes_org ON MICRO_EPISODES(WARD_ID, ORG_NAME)')
    
    conn.commit()
    
    # Clear old episodes for these wards
    cursor.executemany('DELETE FROM MICRO_EPISODES WHERE WARD_ID = ?', [(wid,) for wid in ALL_WARD_IDS])
    conn.commit()
    
    # Insert ALL episodes from episodes_df (alerts + non-alerts)
    episodes_inserted = 0
    for idx, row in episodes_df.iterrows():
        pathogen = REGISTRY.get(row["org_name"])
        if pathogen is None:
            continue
            
        cursor.execute('''
            INSERT INTO MICRO_EPISODES 
            (WARD_ID, ORG_NAME, ORG_ID, START_TIME, END_TIME, NUM_PATIENTS, 
             CULTURE_EVENTS, THRESHOLD, IS_ALERT, DANGER_WEIGHT, WINDOW_DAYS, WARD_SIZE)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            int(row["ward_id"]),
            pathogen.key,
            pathogen.org_id,
            row["start_time"],
            row["end_time"] if pd.notna(row["end_time"]) else None,
            int(row["unique_patients"]),
            int(row["culture_events"]),
            int(row["threshold"]),
            int(row["alert"]),  # 1/0 boolean
            float(pathogen.danger_weight),
            int(row["window_days"]),
            int(row["ward_size"]),
        ))
        episodes_inserted += 1
    
    conn.commit()
    
    # Verify
    total_episodes = cursor.execute('SELECT COUNT(*) FROM MICRO_EPISODES').fetchone()[0]
    alert_count = cursor.execute('SELECT COUNT(*) FROM MICRO_EPISODES WHERE IS_ALERT = 1').fetchone()[0]
    
    print(f"✅ Stored {episodes_inserted} episodes | {alert_count} alerts | Total DB: {total_episodes}")
    
    # Quick DB summary
    summary = pd.read_sql_query('''
        SELECT 
            WARD_ID, 
            COUNT(*) as episodes, 
            SUM(IS_ALERT) as alerts,
            AVG(DANGER_WEIGHT) as avg_severity
        FROM MICRO_EPISODES 
        WHERE WARD_ID IN ({})
        GROUP BY WARD_ID
        ORDER BY alerts DESC
    '''.format(','.join('?' * len(ALL_WARD_IDS))), conn, params=ALL_WARD_IDS)
    print(summary.round(1))


ProgrammingError: Error binding parameter 4: type 'Timestamp' is not supported

In [ ]:
# build a lookup of organism name → org_itemid (int)
microbiologyevents["ORG_NAME"] = (
    microbiologyevents["ORG_NAME"].astype(str)
    .str.upper()
    .str.strip()
)

org_id_map = (
    microbiologyevents
    .dropna(subset=["ORG_NAME", "ORG_ITEMID"])
    .groupby("ORG_NAME")["ORG_ITEMID"]
    .first()          # grab first occurrence for each name
    .astype(int)      # convert float → int
    .to_dict()
)

print(f"{len(org_id_map)} organism names have an ORG_ITEMID; sample:\n",
      list(org_id_map.items())[:8])

# write the ids back into MICROALERTS
with get_db_connection(db_path) as conn:
    cur = conn.cursor()
    for org_name, oid in org_id_map.items():
        cur.execute(
            "UPDATE MICROALERTS SET ORG_ID = ? WHERE ORG_NAME = ?",
            (oid, org_name)
        )
    conn.commit()

    # refresh the dataframe we use for checking
    alerts_verify = pd.read_sql_query(
        'SELECT * FROM MICROALERTS WHERE WARD_ID = ? ORDER BY START_TIME',
        conn,
        params=(WARD_ID_TARGET,)
    )

# optional: keep a local copy named alerts_table too
alerts_table = alerts_verify.copy()

print("after update, alerts table head")
alerts_verify.head()

46 organism names have an ORG_ITEMID; sample:
 [('ACINETOBACTER BAUMANNII', 80194), ('ACINETOBACTER BAUMANNII COMPLEX', 80304), ('ALPHA STREPTOCOCCI', 80052), ('ANAEROBIC GRAM POSITIVE ROD(S)', 80079), ('ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER', 80239), ('BACTEROIDES FRAGILIS GROUP', 80112), ('BETA STREPTOCOCCUS GROUP B', 80045), ('CANDIDA ALBICANS, PRESUMPTIVE IDENTIFICATION', 80254)]
after update, alerts table head


,ALERT_ID,WARD_ID,ORG_ID,ORG_NAME,NUM_PATIENTS,CULTURE_EVENTS,START_TIME,END_TIME,SEVERITY,THRESHOLD,WINDOW_DAYS
0,0,52,80139,CLOSTRIDIUM DIFFICILE,1,1,2105-05-30 00:00:00,2105-06-11 06:57:03,30,1,3
1,2,52,80239,"ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER",1,1,2107-03-23 00:00:00,2107-03-31 06:55:09,15,1,2
2,3,52,80002,ESCHERICHIA COLI,1,28,2119-10-22 00:00:00,2119-11-14 17:03:49,30,1,3
3,4,52,80002,ESCHERICHIA COLI,1,16,2120-08-25 00:00:00,2120-08-25 15:41:49,30,1,3
4,9,52,80293,POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS,1,2,2144-07-14 00:00:00,2144-12-31 21:02:59,30,1,1
